In [ ]:
# Gắn cứng phiên bản numpy dưới 2.0 để sửa lỗi ABI version
# Gắn cứng paddlepaddle-gpu==2.6.1 và paddleocr==2.7.3 để tránh lỗi paddlex
!pip install "numpy<2.0.0" paddlepaddle-gpu==2.6.1 paddleocr==2.7.3


In [ ]:
import os
import json
import glob
import logging
from paddleocr import PaddleOCR
from tqdm import tqdm

# Ẩn log của PaddleOCR 
logging.getLogger('ppocr').setLevel(logging.ERROR)

# 1. Đường dẫn chứa ảnh Dataset Keyframes bạn đã tạo trên Kaggle
# Thay 'your-keyframes-dataset' bằng tên Dataset chứa ảnh của bạn (Ví dụ: /kaggle/input/my-video-keyframes/)
dataset_path = '/kaggle/input/your-keyframes-dataset/'
save_path = '/kaggle/working/ocr_results.json'

if not os.path.exists(dataset_path):
    print(f'THÔNG BÁO: Thư mục {dataset_path} không tồn tại. Vui lòng add Dataset keyframes vào Kaggle qua nút "Add Input".')
    image_files = []
else:
    # Lặp qua các ảnh WebP (định dạng ảnh do TransNetV2/SBD xuất ra)
    image_files = sorted(glob.glob(f"{dataset_path}/**/*.webp", recursive=True))

print(f"Tìm thấy {len(image_files)} ảnh để trích xuất OCR.")

# 2. Khởi tạo PaddleOCR
if image_files:
    print('Loading PaddleOCR...')
    # Bản 2.7.3 quay trở lại dùng use_angle_cls, nếu báo lỗi bạn có thể đổi thành use_textline_orientation=True
    try:
        ocr = PaddleOCR(use_angle_cls=True, lang='vi', show_log=False)
    except Exception:
        ocr = PaddleOCR(use_textline_orientation=True, lang='vi')

results = []

for img_path in tqdm(image_files):
    res = ocr.ocr(img_path, cls=True)
    texts = []
    if res and res[0]:
        for line in res[0]:
            if line and len(line) > 1 and line[1]:
                texts.append(line[1][0])
    
    if texts:
        results.append({
            'frame_name': os.path.basename(img_path),
            'ocr_text': ' '.join(texts)
        })

# Lưu kết quả vào thư mục working của Kaggle để tải về
if results:
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'Đã lưu {len(results)} kết quả tại {save_path}')
elif image_files:
    print('Không tìm thấy chữ nào trong các frames.')
